In [ ]:
!pip install -q transformers datasets scikit-learn tqdm

In [ ]:
import json, random, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, RobertaModel
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from datasets import load_dataset
from tqdm import tqdm

CFG = {
    "name":        "baseline_codebert",
    "hidden_dim":  256,
    "dropout":     0.3,
    "model_name":  "microsoft/codebert-base",
    "epochs":      20,
    "batch_size":  32,
    "lr":          2e-5,
    "max_seq_len": 512,
    "frac":        1.0,
    "seed":        42,
    "early_stop":  6,
    "fp16":        True,
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPUS = torch.cuda.device_count()
print(f"Device : {DEVICE}  |  GPUs : {N_GPUS}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])

In [ ]:
print("Downloading Devign from HuggingFace...")
raw = load_dataset("DetectVul/devign")

def to_rows(split, frac=1.0):
    rows = [{"func": r["func"], "label": float(r["target"])} for r in raw[split]]
    if frac < 1.0:
        rng  = random.Random(CFG["seed"])
        rows = rng.sample(rows, int(len(rows) * frac))
    return rows

train_rows = to_rows("train",      CFG["frac"])
val_rows   = to_rows("validation", 1.0)
test_rows  = to_rows("test",       1.0)
print(f"Train: {len(train_rows)} | Val: {len(val_rows)} | Test: {len(test_rows)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

class DevignTextDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        enc = tokenizer(
            row["func"],
            max_length=CFG["max_seq_len"],
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(row["label"], dtype=torch.float),
        }

train_loader = DataLoader(DevignTextDataset(train_rows), batch_size=CFG["batch_size"], shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(DevignTextDataset(val_rows),   batch_size=CFG["batch_size"], shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(DevignTextDataset(test_rows),  batch_size=CFG["batch_size"], shuffle=False, num_workers=4, pin_memory=True)
print("Dataloaders ready")

In [ ]:
class LLMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder    = RobertaModel.from_pretrained(CFG["model_name"])
        self.proj       = nn.Linear(768, CFG["hidden_dim"])
        self.classifier = nn.Sequential(
            nn.Dropout(CFG["dropout"]),
            nn.Linear(CFG["hidden_dim"], 1),
        )
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(self.proj(cls)).squeeze(-1)

model = LLMClassifier()
if N_GPUS > 1:
    print(f"Using {N_GPUS} GPUs with DataParallel")
    model = nn.DataParallel(model)
model = model.to(DEVICE)

# Uniform LR for all params — avoids head oscillation that tanked the previous run
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)

# Linear warmup (1 epoch) then linear decay to 0 — standard BERT fine-tuning recipe
from transformers import get_linear_schedule_with_warmup
num_training_steps = CFG["epochs"] * len(train_loader)
num_warmup_steps   = len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

criterion = nn.BCEWithLogitsLoss()
scaler    = torch.cuda.amp.GradScaler(enabled=CFG["fp16"])
print(f"Model ready — {sum(p.numel() for p in model.parameters())/1e6:.1f}M params  |  fp16={CFG['fp16']}")
print(f"Scheduler  — {num_warmup_steps} warmup steps → linear decay to 0 over {num_training_steps} total steps")

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=CFG["fp16"]):
                logits = model(ids, mask).cpu()
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()
            all_preds.extend(preds.tolist())
            all_labels.extend(batch["label"].long().tolist())
            all_probs.extend(probs.tolist())
    return {
        "f1":        f1_score(all_labels,        all_preds, zero_division=0),
        "acc":       accuracy_score(all_labels,  all_preds),
        "precision": precision_score(all_labels, all_preds, zero_division=0),
        "recall":    recall_score(all_labels,    all_preds, zero_division=0),
        "auc":       roc_auc_score(all_labels,   all_probs),
    }

In [ ]:
best_f1           = 0.0
epochs_no_improve = 0
start             = time.time()

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Ep {epoch:02d}/{CFG['epochs']}", leave=True)

    for batch in pbar:
        ids    = batch["input_ids"].to(DEVICE)
        mask   = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=CFG["fp16"]):
            logits = model(ids, mask)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    val_m = evaluate(model, val_loader)
    print(f"Epoch {epoch:02d}  loss={total_loss/len(train_loader):.4f}  "
          f"val_F1={val_m['f1']:.4f}  val_Acc={val_m['acc']:.4f}  val_AUC={val_m['auc']:.4f}")

    if val_m["f1"] > best_f1:
        best_f1           = val_m["f1"]
        epochs_no_improve = 0
        torch.save(model.state_dict(), "/kaggle/working/best_model.pt")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= CFG["early_stop"]:
        print(f"Early stopping at epoch {epoch}")
        break

print(f"\nBest val F1: {best_f1:.4f}  |  Time: {(time.time()-start)/60:.1f} min")

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_model.pt", map_location=DEVICE, weights_only=True))
test_m = evaluate(model, test_loader)

print("=" * 55)
print("TEST RESULTS — baseline_codebert (full Devign, 20 epochs)")
print("=" * 55)
print(f"  F1        : {test_m['f1']:.4f}")
print(f"  Accuracy  : {test_m['acc']:.4f}")
print(f"  Precision : {test_m['precision']:.4f}")
print(f"  Recall    : {test_m['recall']:.4f}")
print(f"  AUC-ROC   : {test_m['auc']:.4f}")
print("=" * 55)

with open("/kaggle/working/test_results.json", "w") as f:
    json.dump({"model": "baseline_codebert", "dataset": "devign", **test_m}, f, indent=2)
print("\nSaved: /kaggle/working/test_results.json  (download from Output panel)")